In [1]:
import os
import ast
import argparse
import json
from dataclasses import dataclass
from typing import List, Dict, Any

from pandas.api.types import CategoricalDtype  # not really needed anymore, but harmless
import torch
from torch.utils.data import Dataset

import pandas as pd  # only used if you want to inspect things
from PIL import Image

from datasets import load_dataset  # NEW: Hugging Face Datasets

from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    TrainingArguments,
    Trainer,
    set_seed,
    LogitsProcessor, 
    LogitsProcessorList,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
import bitsandbytes as bnb

In [ ]:
# import torch, os
# from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
# from peft import PeftModel

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
LETTER_VOCAB = ["A", "B", "C", "D"]

VISION_TOKENS = ("<|vision_start|>", "<|vision_end|>", "<|image_pad|>", "<|video_pad|>")

def allowed_letter_token_ids(tok):
    ids = set()
    for L in LETTER_VOCAB:
        for pref in ["", " ", "\n"]:
            pieces = tok(pref + L, add_special_tokens=False).input_ids
            if len(pieces) == 1:
                ids.add(pieces[0])
    if not ids:
        # fallback: just take the first id from encoding of the letter
        for L in LETTER_VOCAB:
            pieces = tok(L, add_special_tokens=False).input_ids
            if len(pieces) >= 1:
                ids.add(pieces[0])
    return sorted(ids)


class AllowOnlyTokens(LogitsProcessor):
    def __init__(self, allowed_ids: List[int]):
        self.allowed_ids = None
        self._ids_list = allowed_ids  # lazy tensor

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if self.allowed_ids is None or self.allowed_ids.device != scores.device:
            self.allowed_ids = torch.tensor(
                self._ids_list, device=scores.device, dtype=torch.long
            )
        mask = torch.full_like(scores, float("-inf"))
        mask.scatter_(1, self.allowed_ids.view(1, -1), scores.index_select(1, self.allowed_ids))
        return mask

In [4]:
def build_seed_prompt(english_prompt: str, options_json: str) -> str:
    """
    Build the text part of the prompt for a single VisualSphinx-Seeds example.
    The image will be attached separately via the vision processor.
    """
    try:
        options = json.loads(options_json)
    except Exception:
        options = {}

    base = (english_prompt or "").strip()
    if not base:
        base = (
            "You are given a visual logic puzzle image with multiple choice answers. "
            "Carefully inspect the image and select the correct answer."
        )

    text = base + "\n\nOptions:\n"
    for k in sorted(options.keys()):
        v = str(options[k])
        text += f"{k}. {v}\n"
    text += "\nLook at the image and choose the correct option.\n"
    text += "Answer with a single letter only (A/B/C/D)."

    return text


In [5]:
class VisualSphinxSeedsDataset(Dataset):
    def __init__(self, hf_dataset, processor: AutoProcessor):
        """
        hf_dataset: a datasets.Dataset object from 'VisualSphinx/VisualSphinx-Seeds'
        processor: the Qwen2.5-VL (or other VLM) processor
        """
        self.ds = hf_dataset
        self.processor = processor

    def __len__(self):
        return len(self.ds)

    def _messages(self, image: Image.Image, prompt_text: str, answer_text: str):
        # Qwen2.5-VL chat format
        user_part = [
            {"type": "text", "text": prompt_text},
            {"type": "image", "image": image},
        ]
        assistant_part = [{"type": "text", "text": answer_text}]
        return [
            {"role": "user", "content": user_part},
            {"role": "assistant", "content": assistant_part},
        ]

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.ds[idx]

        # HF column "image" is actually a list of images (length 1) → pick the first
        img_field = ex["image"]
        if isinstance(img_field, list):
            img_field = img_field[0]
        image = img_field.convert("RGB")

        english_prompt = ex.get("english_prompt", "")
        options_json = ex.get("options", "{}")
        correct_answer = str(ex.get("correct_answer", "")).strip().upper()

        prompt_text = build_seed_prompt(english_prompt, options_json)
        answer_text = correct_answer

        messages_full = self._messages(image, prompt_text, answer_text)
        messages_prompt_only = [{"role": "user", "content": messages_full[0]["content"]}]

        text_full = self.processor.apply_chat_template(messages_full, tokenize=False)
        text_prompt_only = self.processor.apply_chat_template(
            messages_prompt_only, tokenize=False, add_generation_prompt=True
        )

        return {
            "image": image,
            "text_full": text_full,
            "text_prompt": text_prompt_only,
            "label_letter": answer_text,
            "idx": int(ex.get("id", idx)),
        }

In [6]:
@dataclass
class VLDataCollator:
    processor: AutoProcessor  # Qwen2.5-VL processor

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        images = [f["image"] for f in features]
        text_full_list = [f["text_full"] for f in features]
        text_prompt_list = [f["text_prompt"] for f in features]

        batch = self.processor(
            text=text_full_list,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        tok = self.processor.tokenizer
        labels = batch["input_ids"].clone()

        # 1) mask padding
        pad_id = tok.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100

        # 2) mask vision special tokens
        for t in VISION_TOKENS:
            tid = tok.convert_tokens_to_ids(t)
            if tid is not None and tid != -1:
                labels[labels == tid] = -100

        # 3) mask everything before the assistant reply
        for i, tprompt in enumerate(text_prompt_list):
            n = len(tok(tprompt, add_special_tokens=False).input_ids)
            labels[i, :n] = -100

        batch["labels"] = labels
        return batch


In [7]:
def get_model_and_processor(model_name: str, bnb_4bit: bool = True):
    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)

    if bnb_4bit:
        model = AutoModelForVision2Seq.from_pretrained(
            model_name,
            trust_remote_code=True,
            device_map="auto",
            torch_dtype=torch.bfloat16,
            quantization_config=dict(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            ),
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForVision2Seq.from_pretrained(
            model_name,
            trust_remote_code=True,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )

    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        modules_to_save=None,
    )
    model = get_peft_model(model, lora_cfg)
    return model, processor


In [8]:
def run_eval_seeds(model, processor, ds_eval: Dataset, max_samples: int = None):
    model.eval()
    correct = 0
    total = 0
    preds, gts, idxs = [], [], []

    allowed_ids = allowed_letter_token_ids(processor.tokenizer)
    lp = LogitsProcessorList([AllowOnlyTokens(allowed_ids)])

    n = len(ds_eval) if max_samples is None else min(max_samples, len(ds_eval))

    for i in range(n):
        ex = ds_eval[i]

        inputs = processor(
            text=[ex["text_prompt"]],
            images=[ex["image"]],
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.inference_mode():
            gen = model.generate(
                **inputs,
                max_new_tokens=1,
                do_sample=False,
                logits_processor=lp,  # force A–D
            )

        new_tokens = gen[0, inputs["input_ids"].shape[1]:]
        out = processor.tokenizer.decode(new_tokens, skip_special_tokens=True)

        pred_letter = None
        for ch in out:
            if ch in LETTER_VOCAB:
                pred_letter = ch
                break

        gt = ex["label_letter"]
        if pred_letter == gt:
            correct += 1
        total += 1
        preds.append(pred_letter if pred_letter is not None else "")
        gts.append(gt)
        idxs.append(ex["idx"])

    acc = correct / max(1, total)
    return {"accuracy": acc, "preds": preds, "gts": gts, "idxs": idxs}


In [5]:
dataset_seed = 43
dataset_ratio = 41

In [9]:
import argparse
import shlex
import textwrap

def build_parser():
    p = argparse.ArgumentParser()
    p.add_argument("--dataset_name", type=str, default="VisualSphinx/VisualSphinx-Seeds")
    p.add_argument("--train_split", type=str, default="seeds_filted")  # or "all_seeds"
    p.add_argument("--eval_split", type=str, default="all_seeds")      # or "" to disable eval

    p.add_argument("--model_name", required=True)
    p.add_argument("--output_dir", required=True)

    # quality filter (optional)
    p.add_argument("--min_reasonableness", type=int, default=4)
    p.add_argument("--min_readability", type=int, default=4)

    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--num_train_epochs", type=int, default=3)
    p.add_argument("--per_device_train_batch_size", type=int, default=4)
    p.add_argument("--per_device_eval_batch_size", type=int, default=4)
    p.add_argument("--gradient_accumulation_steps", type=int, default=1)
    p.add_argument("--learning_rate", type=float, default=1e-4)
    p.add_argument("--warmup_ratio", type=float, default=0.03)
    p.add_argument("--weight_decay", type=float, default=0.0)
    p.add_argument("--logging_steps", type=int, default=10)
    p.add_argument("--save_strategy", type=str, default="epoch")
    p.add_argument("--evaluation_strategy", type=str, default="epoch")
    p.add_argument("--bf16", action="store_true")
    p.add_argument("--max_train_samples", type=int, default=None)
    p.add_argument("--max_eval_samples", type=int, default=None)
    return p

# Example config you can edit directly in the notebook:
arg_str = textwrap.dedent("""
--model_name Qwen/Qwen2.5-VL-7B-Instruct
--output_dir ./qwen2p5_vl_visualsphinx_seeds_lora
--train_split seeds_filted
--eval_split all_seeds
--min_reasonableness 4
--min_readability 4
--num_train_epochs 2
--per_device_train_batch_size 1
--per_device_eval_batch_size 1
--gradient_accumulation_steps 8
--learning_rate 1e-4
--warmup_ratio 0.03
--weight_decay 0.0
--logging_steps 10
--save_strategy epoch
--evaluation_strategy epoch
--bf16
""").strip()

parser = build_parser()
args = parser.parse_args(shlex.split(arg_str))
args


Namespace(dataset_name='VisualSphinx/VisualSphinx-Seeds', train_split='seeds_filted', eval_split='all_seeds', model_name='Qwen/Qwen2.5-VL-7B-Instruct', output_dir='./qwen2p5_vl_visualsphinx_seeds_lora', min_reasonableness=4, min_readability=4, seed=42, num_train_epochs=2, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0001, warmup_ratio=0.03, weight_decay=0.0, logging_steps=10, save_strategy='epoch', evaluation_strategy='epoch', bf16=True, max_train_samples=None, max_eval_samples=None)

In [10]:
set_seed(args.seed)

# Load HF dataset
raw_train = load_dataset(args.dataset_name, split=args.train_split)
raw_eval = load_dataset(args.dataset_name, split=args.eval_split) if args.eval_split else None

def quality_filter(ex):
    ok = True
    if args.min_reasonableness is not None:
        rs = ex.get("reasonableness_score", None)
        if rs is not None:
            ok = ok and (rs >= args.min_reasonableness)
    if args.min_readability is not None:
        rs = ex.get("readability_score", None)
        if rs is not None:
            ok = ok and (rs >= args.min_readability)
    return ok

raw_train = raw_train.filter(quality_filter)
if raw_eval is not None:
    raw_eval = raw_eval.filter(quality_filter)

if args.max_train_samples:
    raw_train = raw_train.select(range(min(args.max_train_samples, len(raw_train))))
if args.max_eval_samples and raw_eval is not None:
    raw_eval = raw_eval.select(range(min(args.max_eval_samples, len(raw_eval))))

len(raw_train), len(raw_eval) if raw_eval is not None else None


(2326, 3693)

In [11]:
model, processor = get_model_and_processor(args.model_name, bnb_4bit=True)

# Wrap in our dataset class
train_dataset = VisualSphinxSeedsDataset(raw_train, processor)
eval_dataset = VisualSphinxSeedsDataset(raw_eval, processor) if raw_eval is not None else None

collator = VLDataCollator(processor)

training_args = TrainingArguments(
    output_dir=args.output_dir,
    num_train_epochs=args.num_train_epochs,
    per_device_train_batch_size=args.per_device_train_batch_size,
    per_device_eval_batch_size=args.per_device_eval_batch_size,
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    learning_rate=args.learning_rate,
    warmup_ratio=args.warmup_ratio,
    weight_decay=args.weight_decay,
    logging_steps=args.logging_steps,
    save_strategy=args.save_strategy,
    eval_strategy=args.evaluation_strategy,
    bf16=args.bf16,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none",
)

model.config.use_cache = False

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

trainer.train()

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
/home/airlay88/planscape/venv/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

/home/airlay88/planscape/venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/airlay88/planscape/venv/lib/python3.10/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 9.28 GiB. GPU 0 has a total capacity of 23.99 GiB of which 0 bytes is free. Of the allocated memory 30.40 GiB is allocated by PyTorch, and 3.89 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
output_dir = args.output_dir
os.makedirs(output_dir, exist_ok=True)

# Save PEFT adapter
model.save_pretrained(output_dir)
processor.tokenizer.save_pretrained(output_dir)

if eval_dataset is not None:
    eval_out = run_eval_seeds(model, processor, eval_dataset, max_samples=args.max_eval_samples)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")

    with open(os.path.join(output_dir, "visualsphinx_seeds_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"id": idx, "pred": p, "label": g}) + "\n")

    with open(os.path.join(output_dir, "visualsphinx_seeds_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

In [10]:
args.image_root

'./datasets/easy_anyshape41_43_mix'

In [11]:


# Datasets
train_dataset = OddOneOutDataset(train_df, args.image_root, processor, legend=args.legend_in_prompt)
eval_dataset = OddOneOutDataset(eval_df, args.image_root, processor, legend=args.legend_in_prompt)

collator = VLDataCollator(processor)

training_args = TrainingArguments(
    output_dir=args.output_dir,
    num_train_epochs=args.num_train_epochs,
    per_device_train_batch_size=args.per_device_train_batch_size,
    per_device_eval_batch_size=args.per_device_eval_batch_size,
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    learning_rate=args.learning_rate,
    warmup_ratio=args.warmup_ratio,
    weight_decay=args.weight_decay,
    logging_steps=args.logging_steps,
    save_strategy=args.save_strategy,
    eval_strategy=args.evaluation_strategy,
    bf16=args.bf16,
    dataloader_pin_memory=False,
    remove_unused_columns=False,  
    report_to="none",
)

model.config.use_cache = False

In [12]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)


In [13]:
# Train
train_result = trainer.train()
trainer.save_model(args.output_dir)
if trainer.is_world_process_zero():
    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)
    trainer.save_state()

# Quick evaluation pass with greedy decoding
if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, args.image_root, args.legend_in_prompt)   
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(args.output_dir, "eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(args.output_dir, "eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", args.output_dir)

/home/airlay88/planscape/venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/airlay88/planscape/venv/lib/python3.10/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.691200,0.155697
2,0.309100,0.131529


/home/airlay88/planscape/venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/airlay88/planscape/venv/lib/python3.10/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


***** train metrics *****
  epoch                    =         2.0
  total_flos               = 308729935GF
  train_loss               =      1.2176
  train_runtime            =  1:55:20.01
  train_samples_per_second =       1.153
  train_steps_per_second   =       0.144


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.7542 on 1009 examples
Done.
Adapter weights are saved under: ./qwen2p5_vl_odd1out_easy_anyshape41_43


In [7]:
df = pd.read_csv("./difficult_dataset_mid_train_test.csv")
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

if max_train_samples:
    train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

NameError: name 'max_train_samples' is not defined

In [19]:
# Quick evaluation pass with greedy decoding
if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, args.image_root, args.legend_in_prompt)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(args.output_dir, "difficult_on_easy_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(args.output_dir, "difficult_on_easy_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", args.output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8903 on 1030 examples
Done.
Adapter weights are saved under: ./qwen2p5_vl_odd1out


In [18]:
import torch, os
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import PeftModel

BASE = "Qwen/Qwen2.5-VL-7B-Instruct"          # or your base path
ADAPTER = "qwen2p5_vl_odd1out"  # where Trainer.save_model() wrote adapters

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(BASE, trust_remote_code=True)

# load base in 4-bit, then attach LoRA
model = AutoModelForVision2Seq.from_pretrained(
    BASE,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)
model = PeftModel.from_pretrained(model, ADAPTER, is_trainable=False)
model.eval()

/home/airlay88/planscape/venv/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2199: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2_5_VLForConditionalGeneration(
      (model): Qwen2_5_VLModel(
        (visual): Qwen2_5_VisionTransformerPretrainedModel(
          (patch_embed): Qwen2_5_VisionPatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-31): 32 x Qwen2_5_VLVisionBlock(
              (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
              (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
              (attn): Qwen2_5_VLVisionAttention(
                (qkv): Linear4bit(in_features=1280, out_features=3840, bias=True)
                (proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
              )
              (mlp): Qwen2_5_VLMLP(
                (gate_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=1280, out_features=3420, bias=True)

In [19]:
df = pd.read_csv("./difficult_dataset_mid_train_test.csv")
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

# if max_train_samples:
#     train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

/tmp/ipykernel_344332/1119426447.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['orig_index'] += 1
/tmp/ipykernel_344332/1119426447.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df['orig_index'] += 1


In [20]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/difficult_mix"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_difficult_mix_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_difficult_mix_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.4986 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [21]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/difficult_reoriented"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_difficult_reoriented_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_difficult_reoriented_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.5697 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [22]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/difficult_original"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_difficult_original_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_difficult_original_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6152 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [23]:
df = pd.read_csv("./easy_dataset_mid_train_test.csv")
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

# if max_train_samples:
#     train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

/tmp/ipykernel_344332/1731702007.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['orig_index'] += 1
/tmp/ipykernel_344332/1731702007.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df['orig_index'] += 1


In [24]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/easy_mix"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_easy_mix_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_easy_mix_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6777 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [25]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/easy_original"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_easy_original_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_easy_original_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6534 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [26]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/easy_reoriented"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_easy_reoriented_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_easy_reoriented_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.7922 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [6]:
BASE = "Qwen/Qwen2.5-VL-7B-Instruct"          # or your base path
ADAPTER = "qwen2p5_vl_odd1out_easy"  # where Trainer.save_model() wrote adapters

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(BASE, trust_remote_code=True)

# load base in 4-bit, then attach LoRA
model = AutoModelForVision2Seq.from_pretrained(
    BASE,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)
model = PeftModel.from_pretrained(model, ADAPTER, is_trainable=False)
model.eval()

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
/home/airlay88/planscape/venv/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2199: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2_5_VLForConditionalGeneration(
      (model): Qwen2_5_VLModel(
        (visual): Qwen2_5_VisionTransformerPretrainedModel(
          (patch_embed): Qwen2_5_VisionPatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-31): 32 x Qwen2_5_VLVisionBlock(
              (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
              (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
              (attn): Qwen2_5_VLVisionAttention(
                (qkv): Linear4bit(in_features=1280, out_features=3840, bias=True)
                (proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
              )
              (mlp): Qwen2_5_VLMLP(
                (gate_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=1280, out_features=3420, bias=True)

In [7]:
df = pd.read_csv("./difficult_dataset_mid_train_test.csv")
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

# if max_train_samples:
#     train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

/tmp/ipykernel_374897/1119426447.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['orig_index'] += 1
/tmp/ipykernel_374897/1119426447.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df['orig_index'] += 1


In [10]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/difficult_mix"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_difficult_mix_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_difficult_mix_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6550 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [11]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/difficult_original"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_difficult_original_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_difficult_original_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6171 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [12]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/difficult_reoriented"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_difficult_reoriented_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_difficult_reoriented_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8512 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [13]:
df = pd.read_csv("./easy_dataset_mid_train_test.csv")
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

# if max_train_samples:
#     train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

/tmp/ipykernel_374897/1731702007.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['orig_index'] += 1
/tmp/ipykernel_374897/1731702007.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df['orig_index'] += 1


In [14]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_mix"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_mix_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_mix_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8573 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [15]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_reoriented"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_reoriented_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_reoriented_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.9369 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [16]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_original"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_original_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_original_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8408 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [16]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/difficult_mix"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, True)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_difficult_mix_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_difficult_mix_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6559 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [19]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_mix"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, True)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_mix_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_mix_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8621 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [20]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_original"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, True)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_original_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_original_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8437 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [21]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_reoriented"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, True)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_reoriented_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_reoriented_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.9456 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [ ]:
image_root = "./datasets/easy"
output_dir = "qwen2p5_zeroshot"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, True)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "zeroshot_on_easy_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "zeroshot_on_easy_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.2039 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_zeroshot
